# Anthropic-style J-lens coordinate swap — MOCK only

This notebook builds and checks the **coordinate-swap** intervention on a
synthetic world. It **never loads Gemma**, never touches Drive, never reads a
completed run, and never produces a scientific result. Opening it and running
every cell top to bottom starts nothing expensive.

## The two interventions this repository now contains

They are different measurements and must never share an artifact.

**A — Completed pilot: source-derived J-space steering.** The method behind
the finished three-modality causal result (`jlens.mmpilot.causal`). A concept
direction is estimated *entirely from source-modality training examples* as
`delta = ReLU(mean positive code - mean negative code)`, mapped back through
the frozen dictionary as `v_concept = V delta`, unit-normalized, and then
added or subtracted:

$$h' = h \pm \alpha\, v_{\text{concept}}$$

That is a valid causal steering experiment, and it is what the completed
result measured. It is **not** a coordinate swap.

**B — Planned identity replacement: exact two-coordinate patching.** The
paper's method (*Technical details of J-lens use cases*,
https://transformer-circuits.pub/2026/workspace/index.html). Take the two
tokens' J-lens vectors as the **columns** of a matrix, read the activation's
coordinates in that basis with a pseudoinverse, and exchange them:

$$V = [\,v_{\text{source}}\;\; v_{\text{target}}\,], \qquad
c = V^{\dagger} h, \qquad
h_{\text{patched}} = h + \alpha\, V\,(\sigma(c) - c)$$

At `alpha = 1` this is a *measured* exchange, and the component of `h`
orthogonal to `span{v_source, v_target}` is unchanged exactly.

**B is not `h + alpha (v_target - v_source)`.** The update direction is
parallel to `v_source - v_target`, but its coefficient is `c_target -
c_source`, read off the activation. An activation carrying no source content
is barely touched; one saturated with it is rewritten. That measured
coefficient is the whole method.

## What the planned experiment will and will not claim

* Behavioral outputs stay **text**. `text`, `image` and `spoken_audio` are
  *evidence* modalities.
* `spoken_audio` means **spoken captions**, not environmental audio.
* **Identity replacement** and **downstream recomputation** are separate
  claims, tested separately, with separate controls.
* Nothing about generalization, multi-hop reasoning, or multimodal transfer
  under method B is claimed anywhere until those real experiments are run.


## 1. Colab bootstrap

Constants only in the first cell; nothing from this repository is imported
until it has been installed.


In [ ]:
# 1a. Bootstrap constants only. Nothing from this repository is imported yet.
REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
BRANCH = "experiment/spokencoco-jspace-pilot"
REPO_DIR = "/content/jacobian-lens-gemma"

print(f"repo   {REPO_URL}")
print(f"branch {BRANCH}")
print(f"target {REPO_DIR}")


In [ ]:
# 1b. Clone or update the repository, then verify the checked-out branch.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_PATH = Path(os.environ.get("MMPILOT_REPO_DIR") or REPO_DIR)


def _git(*arguments, cwd=None):
    result = subprocess.run(
        ["git", *arguments], cwd=cwd, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(arguments)} failed:\n{result.stdout}\n{result.stderr}"
        )
    return result.stdout.strip()


if IN_COLAB:
    if not (REPO_PATH / ".git").is_dir():
        _git("clone", "--branch", BRANCH, REPO_URL, str(REPO_PATH))
    else:
        _git("fetch", "origin", BRANCH, cwd=REPO_PATH)
        _git("checkout", BRANCH, cwd=REPO_PATH)
        _git("reset", "--hard", f"origin/{BRANCH}", cwd=REPO_PATH)

CHECKED_OUT_BRANCH = _git("rev-parse", "--abbrev-ref", "HEAD", cwd=REPO_PATH)
COMMIT = _git("rev-parse", "HEAD", cwd=REPO_PATH)
if IN_COLAB and CHECKED_OUT_BRANCH != BRANCH:
    raise RuntimeError(
        f"checked out {CHECKED_OUT_BRANCH!r}, expected {BRANCH!r} - "
        "refusing to continue against the wrong code"
    )
print(f"branch {CHECKED_OUT_BRANCH}")
print(f"commit {COMMIT}")


In [ ]:
# 1c. Install the repository and verify that `import jlens` resolves here.
if IN_COLAB:
    print("installing the repository (editable) ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_PATH)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"pip install -e failed:\n{result.stdout[-2000:]}\n{result.stderr[-2000:]}"
        )

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

try:
    import jlens
except ModuleNotFoundError as error:
    raise RuntimeError(
        f"`import jlens` is still not importable after installation: {error}"
    ) from error

if Path(jlens.__file__).resolve().parent.parent != REPO_PATH.resolve():
    raise RuntimeError(f"`import jlens` resolved to {jlens.__file__}, not this checkout")
print(f"jlens  {jlens.__file__}")
print(f"cwd    {os.getcwd()}")


## 2. The switch, and why it refuses

There is exactly one switch and it is `False` in the committed notebook. Unlike
the other notebooks in this repository, setting it to `True` does **not** start
a real experiment — it raises, with the reason. The real coordinate-swap study
cannot be designed yet: the paper applies the swap over a *contiguous band of
intermediate layers*, and the published calibration confirmed three isolated
layers (35, 38, 40), not a band. Which band exists is what the running
earlier-layer calibration decides.


In [ ]:
# 2. The one switch. False in the committed notebook, and True is a refusal.
RUN_REAL_COORDINATE_SWAP = False

if RUN_REAL_COORDINATE_SWAP:
    raise NotImplementedError(
        "The real coordinate-swap experiment does not exist yet, and this "
        "notebook will not improvise one.\n\n"
        "Blocking reason: the paper's protocol applies the swap at every token "
        "position over a CONTIGUOUS band of intermediate layers. The completed "
        "research-grade calibration confirmed layers 35, 38 and 40 on its "
        "untouched confirmation set - three isolated layers, not a band. Layer "
        "32 and every earlier tested layer FAILED confirmation.\n\n"
        "jlens.mmpilot.coordinate_swap.build_layer_band() refuses a band "
        "containing an unconfirmed layer, so there is currently no admissible "
        "band to run. The earlier-layer calibration now in progress is what "
        "decides whether one exists.\n\n"
        "Until then this notebook is MOCK-only by construction."
    )

MODE = "mock"
print(f"mode {MODE} - no model, no Drive, no completed run is read or written")


In [ ]:
# 3. Imports. Everything here is CPU-only and deterministic.
import json
import shutil
import tempfile

import torch

from jlens.interventions import residual_intervention
from jlens.mmpilot import causal
from jlens.mmpilot.backend import run_invariance_gate
from jlens.mmpilot.capability import prediction_and_margin, score_candidate_sequences
from jlens.mmpilot.coordinate_swap import (
    CONTROL_KINDS,
    INTERVENTION_FAMILY,
    METHOD_VERSION,
    NORMALIZATION_CONVENTION,
    POSITION_RULES,
    PRIMARY_POSITION_RULE,
    PROMPT_BOUNDARY_RULE,
    SOLVE_POLICY,
    VECTOR_CONVENTION,
    ConceptToken,
    CoordinateSwapError,
    IllConditionedPairError,
    LayerBandError,
    MultiTokenConceptError,
    RankDeficientPairError,
    StabilityPolicy,
    assert_coordinate_swap_artifacts,
    assert_vector_orientation,
    basis_diagnostics,
    build_layer_band,
    build_spec,
    build_swap_basis,
    coordinate_swap_band,
    coordinate_swap_fingerprint,
    direct_answer_vector,
    orthogonal_residual,
    random_two_direction_basis,
    read_coordinates,
    resolve_concept_token,
    resolve_positions,
    reverse_basis,
    run_swap_condition,
    swap_coordinates,
)
from jlens.mmpilot.coordinate_swap_mock import (
    IDENTITY_CANDIDATES,
    IDENTITY_QUESTION,
    MOCK_LENS_CHECKSUM,
    MOCK_MODALITIES,
    MOCK_MODEL_REVISION,
    MOCK_PROCESSOR_REVISION,
    MOCK_VALIDATED_LAYERS,
    PARALLEL_ID,
    POST_REASONING_BAND,
    PRIMARY_BAND,
    PROPERTY_CANDIDATES,
    PROPERTY_QUESTION,
    REASONING_LAYER,
    SwapMockBackend,
    band_parity_diagnostic,
    mock_bases,
    mock_concept_tokens,
    mock_lens_checksums,
)
from jlens.mmpilot.published_lens import CONFIRMED_LAYERS, FAILED_CONFIRMATION_LAYERS
from jlens.mmpilot.store import IncompatibleStateError, RunFingerprint, UnitStore

torch.manual_seed(0)
print(f"method  {METHOD_VERSION}")
print(f"family  {INTERVENTION_FAMILY}")
print(f"vectors {VECTOR_CONVENTION}")
print(f"norm    {NORMALIZATION_CONVENTION}")
print(f"solve   {SOLVE_POLICY}")


## 4. A synthetic, deliberately nonorthogonal J-lens pair

The whole method turns on `pinv(V) h` not being `V^T h`. A mock whose lens
vectors were orthonormal would hide that, so this pair is built with a stated
cosine of 0.45 and stated norms of 0.8 and 1.3 — and the identity readout rows
are the rows of `pinv(V)`, so a logit *is* a lens coordinate and every expected
value below is closed-form.


In [ ]:
# 4a. Build the synthetic world and resolve each concept to exactly one token.
BACKEND = SwapMockBackend()
WORLD = BACKEND.world
TOKENS = mock_concept_tokens(BACKEND)

for name, token in TOKENS.items():
    print(f"{name:>6} -> token {token.token_id:>3}  text {token.token_text!r}  variant {token.variant!r}")

SOURCE, TARGET = TOKENS["bird"], TOKENS["cat"]
print()
print(f"source {SOURCE.concept!r} id {SOURCE.token_id}")
print(f"target {TARGET.concept!r} id {TARGET.token_id}")


In [ ]:
# 4b. One basis per band layer, with its rank / condition diagnostics.
BASES = mock_bases(WORLD, layers=PRIMARY_BAND, source=SOURCE, target=TARGET)
BASIS_DIAGNOSTICS = {layer: basis.diagnostics for layer, basis in BASES.items()}

for layer, diagnostics in BASIS_DIAGNOSTICS.items():
    print(
        f"L{layer}: |v_source|={diagnostics['source_norm']:.4f} "
        f"|v_target|={diagnostics['target_norm']:.4f} "
        f"cos={diagnostics['cosine']:.4f} "
        f"rank={diagnostics['numerical_rank']} "
        f"cond={diagnostics['condition_number']:.4f}"
    )
print()
print("V checksum L%d: %s" % (PRIMARY_BAND[0], BASIS_DIAGNOSTICS[PRIMARY_BAND[0]]['V_checksum']))


In [ ]:
# 4c. The coordinates are NOT the dot products. Shown, not asserted.
h_pure_source = 2.0 * WORLD.v_source
V = WORLD.V
COORDINATE_VS_DOT = {
    "coordinates_pinv": [float(x) for x in read_coordinates(h_pure_source, V)],
    "dot_products": [float(x) for x in (V.T @ h_pure_source)],
}
print("h = 2 * v_source")
print(f"  pinv(V) h    = {COORDINATE_VS_DOT['coordinates_pinv']}   <- the coordinates")
print(f"  V^T h        = {COORDINATE_VS_DOT['dot_products']}   <- NOT the coordinates")
assert abs(COORDINATE_VS_DOT["coordinates_pinv"][1]) < 1e-12
assert abs(COORDINATE_VS_DOT["dot_products"][1]) > 0.1
print()
print("Exchanging the coordinates lands exactly on 2 * v_target:")
patched_pure, _ = swap_coordinates(h_pure_source, V, alpha=1.0)
print(f"  max |h_patched - 2 v_target| = {float((patched_pure - 2.0 * WORLD.v_target).abs().max()):.3e}")


## 5. What the implementation refuses

Each of these would otherwise produce a plausible-looking number meaning
something other than what it would be reported as.


In [ ]:
# 5. Every refusal, exercised.
REFUSALS = {}


def _refused(label, exception_type, thunk):
    try:
        thunk()
    except exception_type as error:
        REFUSALS[label] = f"{type(error).__name__}: {str(error).splitlines()[0]}"
        return
    raise AssertionError(f"{label} was NOT refused")


ATOMS = WORLD.atoms()
_PARALLEL = ConceptToken("parallel", PARALLEL_ID, " parallel", " {}")

_refused(
    "rank_deficient_pair",
    RankDeficientPairError,
    lambda: build_swap_basis(ATOMS, layer=1, source=SOURCE, target=_PARALLEL),
)

_near = torch.zeros(20, WORLD.d_model, dtype=torch.float64)
_near[0] = WORLD.Q[:, 0]
_near[1] = (1 - 1e-9) * WORLD.Q[:, 0] + (1 - (1 - 1e-9) ** 2) ** 0.5 * WORLD.Q[:, 1]
_refused(
    "ill_conditioned_pair",
    IllConditionedPairError,
    lambda: build_swap_basis(
        _near, layer=1, source=ConceptToken("a", 0, " a", " {}"),
        target=ConceptToken("b", 1, " b", " {}"),
    ),
)

_refused(
    "transposed_basis",
    CoordinateSwapError,
    lambda: assert_vector_orientation(V.T, d_model=WORLD.d_model),
)

_refused(
    "multi_token_concept",
    MultiTokenConceptError,
    lambda: resolve_concept_token(lambda text: [1, 2, 3], "strawberry"),
)

_refused(
    "identical_source_and_target",
    CoordinateSwapError,
    lambda: build_swap_basis(ATOMS, layer=1, source=SOURCE, target=SOURCE),
)

_refused(
    "unvalidated_layer_in_band",
    LayerBandError,
    lambda: build_layer_band(1, 4, validated_layers=MOCK_VALIDATED_LAYERS),
)

_refused(
    "position_rule_without_evidence_span",
    CoordinateSwapError,
    lambda: resolve_positions("evidence_span_only", prompt_len=4, seq_len=4),
)

for label, message in REFUSALS.items():
    print(f"refused  {label}")
    print(f"         {message[:150]}")


In [ ]:
# 5b. The band gate against the REAL published record, not a notebook literal.
print(f"confirmed layers (published calibration): {CONFIRMED_LAYERS}")
print(f"failed confirmation:                      {FAILED_CONFIRMATION_LAYERS}")
try:
    build_layer_band(35, 38, validated_layers=CONFIRMED_LAYERS, n_layers=42)
except LayerBandError as error:
    REAL_BAND_REFUSAL = str(error).splitlines()[0]
    print()
    print("A contiguous L35-L38 band is refused today:")
    print(f"  {REAL_BAND_REFUSAL}")
else:  # pragma: no cover - would mean the record changed
    raise AssertionError("a contiguous band over 35..38 should not be admissible")


## 6. The algebra, checked numerically

`alpha = 0` is bit-exact; `alpha = 1` exchanges the coordinates; two swaps
cancel; the orthogonal component never moves; `alpha = 2` is extrapolation, not
a stronger swap.


In [ ]:
# 6. Algebra checks on a random activation.
_generator = torch.Generator().manual_seed(31)
h = torch.randn(WORLD.d_model, generator=_generator, dtype=torch.float64)

zero, _ = swap_coordinates(h, V, alpha=0.0)
once, record_once = swap_coordinates(h, V, alpha=1.0)
twice, _ = swap_coordinates(once, V, alpha=1.0)

before = read_coordinates(h, V)
after = read_coordinates(once, V)
residual_before = orthogonal_residual(h, V)
residual_after = orthogonal_residual(once, V)

ALGEBRA = {
    "alpha_zero_is_bit_exact": bool(torch.equal(zero, h)),
    "double_swap_recovers": float((twice - h).abs().max()),
    "coordinates_before": [float(x) for x in before],
    "coordinates_after": [float(x) for x in after],
    "coordinates_exchanged": float((after - before.flip(0)).abs().max()),
    "orthogonal_drift": float((residual_after - residual_before).abs().max()),
    "recorded_orthogonal_drift": record_once["max_orthogonal_residual_drift"],
}
for alpha in (0.0, 0.5, 1.0, 2.0):
    patched, record = swap_coordinates(h, V, alpha=alpha)
    ALGEBRA[f"coordinates_at_alpha_{alpha}"] = [
        float(x) for x in read_coordinates(patched, V)
    ]
    ALGEBRA[f"extrapolation_at_alpha_{alpha}"] = record["alpha_is_extrapolation"]

assert ALGEBRA["alpha_zero_is_bit_exact"]
assert ALGEBRA["double_swap_recovers"] < 1e-12
assert ALGEBRA["coordinates_exchanged"] < 1e-12
assert ALGEBRA["orthogonal_drift"] < 1e-12

print(f"c before                {ALGEBRA['coordinates_before']}")
print(f"c after  (alpha=1)      {ALGEBRA['coordinates_after']}")
print(f"c after  (alpha=0.5)    {ALGEBRA['coordinates_at_alpha_0.5']}   <- interpolation")
print(f"c after  (alpha=2)      {ALGEBRA['coordinates_at_alpha_2.0']}   <- EXTRAPOLATION, not a swap")
print()
print(f"alpha=0 bit-exact no-op        {ALGEBRA['alpha_zero_is_bit_exact']}")
print(f"double swap recovers h         {ALGEBRA['double_swap_recovers']:.3e}")
print(f"orthogonal component drift     {ALGEBRA['orthogonal_drift']:.3e}")


## 7. The same concept, arriving through three evidence channels

Every input is byte-identical apart from its evidence channel. In this
synthetic world the shared concept vector *is* shared by construction — which
is exactly why a MOCK pass says nothing about whether it is shared in Gemma.


In [ ]:
# 7a. Build one identity input and one property input per modality.
IDENTITY_IDS = {name: BACKEND.encode_candidate(f" {name}") for name in IDENTITY_CANDIDATES}
PROPERTY_IDS = {name: BACKEND.encode_candidate(f" {name}") for name in PROPERTY_CANDIDATES}

INPUTS = {}
CLEAN = {}
for modality in MOCK_MODALITIES:
    for question, ids, kind in (
        (IDENTITY_QUESTION, IDENTITY_IDS, "identity"),
        (PROPERTY_QUESTION, PROPERTY_IDS, "property"),
    ):
        built = BACKEND.build_inputs(prompt=question, modality=modality, concept="bird")
        INPUTS[(modality, kind)] = built
        CLEAN[(modality, kind)] = score_candidate_sequences(BACKEND, built, ids)

BASELINE = {}
for (modality, kind), scores in CLEAN.items():
    target = "cat" if kind == "identity" else "four"
    BASELINE[f"{modality}/{kind}"] = prediction_and_margin(scores, target)["prediction"]

_probe = INPUTS[("image", "identity")]
print(f"prompt_len {_probe.prompt_len}, evidence span {_probe.modality_token_range}")
print(f"candidates are multi-token: {[len(v) for v in IDENTITY_IDS.values()]}")
print()
for key, prediction in BASELINE.items():
    print(f"clean {key:<26} -> {prediction}")
assert set(BASELINE.values()) == {"bird", "two"}


In [ ]:
# 7b. The mandatory invariance gate, on the coordinate-swap band's layers.
INVARIANCE = run_invariance_gate(BACKEND, INPUTS[("image", "identity")], list(PRIMARY_BAND))
print(f"capture hook is a no-op        max|dlogit| = {INVARIANCE['capture_noop']['max_abs_logit_diff']:.3e}")
for row in INVARIANCE["zero_intervention"]:
    print(f"zero-coefficient edit at L{row['layer']}   max|dlogit| = {row['max_abs_logit_diff']:.3e}")
assert INVARIANCE["passed"]


## 8. The paper's protocol: every prompt position, over a layer band

`all_prompt_positions` is the primary rule. It includes the multimodal
evidence positions, and it never includes a teacher-forced candidate token —
that boundary is enforced inside the hook, for every rule.


In [ ]:
# 8a. Where the swap lands, and where it provably does not.
BAND = build_layer_band(
    min(PRIMARY_BAND), max(PRIMARY_BAND), validated_layers=MOCK_VALIDATED_LAYERS, n_layers=BACKEND.n_layers
)
print(f"band {BAND.layers} (contiguous, every layer validated)")
print(f"boundary rule: {PROMPT_BOUNDARY_RULE}")
print()

_inputs = INPUTS[("image", "identity")]
_seq_len = _inputs.prompt_len + max(len(v) for v in IDENTITY_IDS.values())
POSITION_SELECTION = {
    rule: resolve_positions(
        rule,
        prompt_len=_inputs.prompt_len,
        seq_len=_seq_len,
        evidence_span=_inputs.modality_token_range,
    )
    for rule in POSITION_RULES
}
for rule, positions in POSITION_SELECTION.items():
    marker = "  <- PRIMARY" if rule == PRIMARY_POSITION_RULE else ""
    print(f"{rule:<32} {len(positions):>2} positions {positions}{marker}")
    assert max(positions) < _inputs.prompt_len
print()
print(f"teacher-forced candidate positions {list(range(_inputs.prompt_len, _seq_len))} are never patched")


In [ ]:
# 8b. The identity swap, at every prompt position, over the whole band,
#     in each of the three evidence modalities.
IDENTITY_RESULTS = {}
for modality in MOCK_MODALITIES:
    IDENTITY_RESULTS[modality] = run_swap_condition(
        BACKEND,
        INPUTS[(modality, "identity")],
        bases=BASES,
        alpha=1.0,
        candidate_ids=IDENTITY_IDS,
        target_concept="cat",
        clean_scores=CLEAN[(modality, "identity")],
        position_rule=PRIMARY_POSITION_RULE,
        record_coordinates=True,
    )

for modality, result in IDENTITY_RESULTS.items():
    print(
        f"{modality:<13} {result['clean_prediction']} -> {result['prediction']}   "
        f"layers {result['layers_patched']}  positions {result['n_positions_patched']}  "
        f"candidate positions skipped {result['n_candidate_positions_skipped']}"
    )
assert all(r["prediction"] == "cat" for r in IDENTITY_RESULTS.values())


In [ ]:
# 8c. Each layer recomputed its own coordinates - nothing was replayed.
_stats = IDENTITY_RESULTS["image"]["layer_stats"]
PER_LAYER_COORDINATES = {
    layer: row["swap"]["coordinates_before"][_inputs.modality_token_range[0]]
    for layer, row in _stats.items()
}
for layer, coordinates in PER_LAYER_COORDINATES.items():
    print(f"L{layer} pre-swap c at the first evidence position = [{coordinates[0]:+.5f}, {coordinates[1]:+.5f}]")
assert len({tuple(c) for c in PER_LAYER_COORDINATES.values()}) == len(PER_LAYER_COORDINATES)
print()
print("distinct at every layer: the coordinates are read from that layer's own activation")


## 9. Downstream recomputation — a separate claim, tested separately

The same bird -> cat patch, a different question. In this world the legs answer
is *computed* at the reasoning layer from whatever identity coordinates reach
it, so a band before that layer changes it and a band after it cannot.


In [ ]:
# 9a. Same swap, different question.
PROPERTY_RESULTS = {}
for modality in MOCK_MODALITIES:
    PROPERTY_RESULTS[modality] = run_swap_condition(
        BACKEND,
        INPUTS[(modality, "property")],
        bases=BASES,
        alpha=1.0,
        candidate_ids=PROPERTY_IDS,
        target_concept="four",
        clean_scores=CLEAN[(modality, "property")],
        position_rule=PRIMARY_POSITION_RULE,
    )
for modality, result in PROPERTY_RESULTS.items():
    print(f"{modality:<13} legs: {result['clean_prediction']} -> {result['prediction']}")
assert all(r["prediction"] == "four" for r in PROPERTY_RESULTS.values())


In [ ]:
# 9b. The layer-band control: the same swap, applied after the property was
#     already computed. Identity moves; the property does not.
POST_BASES = mock_bases(WORLD, layers=POST_REASONING_BAND, source=SOURCE, target=TARGET)
LAYER_BAND_CONTROL = {
    "identity": run_swap_condition(
        BACKEND, INPUTS[("image", "identity")], bases=POST_BASES, alpha=1.0,
        candidate_ids=IDENTITY_IDS, target_concept="cat",
        clean_scores=CLEAN[("image", "identity")],
    )["prediction"],
    "property": run_swap_condition(
        BACKEND, INPUTS[("image", "property")], bases=POST_BASES, alpha=1.0,
        candidate_ids=PROPERTY_IDS, target_concept="four",
        clean_scores=CLEAN[("image", "property")],
    )["prediction"],
}
print(f"band {PRIMARY_BAND} (before the reasoning layer L{REASONING_LAYER}): identity -> cat, legs -> four")
print(f"band {POST_REASONING_BAND} (from its output onward):        identity -> {LAYER_BAND_CONTROL['identity']}, legs -> {LAYER_BAND_CONTROL['property']}")
assert LAYER_BAND_CONTROL == {"identity": "cat", "property": "two"}
print()
print("Identity replacement without downstream recomputation is observable.")
print("They are separate claims and must be reported separately.")


## 10. The controls

Every control runs through the same code path as the condition under test.
The **direct-answer-vector** control is the one that decides whether a
downstream change may be called recomputation at all: if simply inserting the
answer's own lens vector at the same depth moves the answer just as well, the
swap's downstream effect is not evidence that anything was re-derived.


In [ ]:
# 10a. Coordinate-swap controls: zero, norm-matched random, unrelated pair,
#      reverse swap, and the two position controls.
_identity_inputs = INPUTS[("image", "identity")]
_identity_clean = CLEAN[("image", "identity")]


def _swap_prediction(bases, *, alpha=1.0, position_rule=PRIMARY_POSITION_RULE, inputs=None, clean=None, target="cat"):
    return run_swap_condition(
        BACKEND,
        inputs if inputs is not None else _identity_inputs,
        bases=bases,
        alpha=alpha,
        candidate_ids=IDENTITY_IDS,
        target_concept=target,
        clean_scores=clean if clean is not None else _identity_clean,
        position_rule=position_rule,
    )["prediction"]


RANDOM_BASES = {
    layer: random_two_direction_basis(basis, seed=1000 + layer)
    for layer, basis in BASES.items()
}
UNRELATED_BASES = mock_bases(WORLD, layers=PRIMARY_BAND, source=TOKENS["dog"], target=TOKENS["car"])

_cat_inputs = BACKEND.build_inputs(prompt=IDENTITY_QUESTION, modality="spoken_audio", concept="cat")
_cat_clean = score_candidate_sequences(BACKEND, _cat_inputs, IDENTITY_IDS)
REVERSE_BASES = {layer: reverse_basis(basis) for layer, basis in BASES.items()}

CONTROLS = {
    "coordinate_swap": _swap_prediction(BASES),
    "zero": _swap_prediction(BASES, alpha=0.0),
    "random_two_direction_norm_matched": _swap_prediction(RANDOM_BASES),
    "unrelated_pair_swap": _swap_prediction(UNRELATED_BASES),
    "position_control": _swap_prediction(BASES, position_rule="non_evidence_prompt_positions"),
    "final_prompt_token_only_comparison": _swap_prediction(BASES, position_rule="final_prompt_token_only"),
    "reverse_swap": _swap_prediction(
        REVERSE_BASES, inputs=_cat_inputs, clean=_cat_clean, target="bird"
    ),
    "layer_band_control": LAYER_BAND_CONTROL["identity"],
}
for name, prediction in CONTROLS.items():
    print(f"{name:<38} -> {prediction}")

assert CONTROLS["coordinate_swap"] == "cat"
assert CONTROLS["zero"] == "bird"
assert CONTROLS["random_two_direction_norm_matched"] == "bird"
assert CONTROLS["unrelated_pair_swap"] == "bird"
assert CONTROLS["position_control"] == "bird"
assert CONTROLS["reverse_swap"] == "bird"
print()
print("`final_prompt_token_only` is the COMPLETED pilot's rule, kept only as a")
print("labelled comparison. Here it does not reproduce the paper's protocol,")
print("which is the point of running both.")


In [ ]:
# 10b. The direct-answer-vector control. This is the essential one.
_property_inputs = INPUTS[("image", "property")]
_property_clean = CLEAN[("image", "property")]
_answer_delta = direct_answer_vector(ATOMS, answer_token_id=TOKENS["four"].token_id, scale=8.0)

with residual_intervention(
    BACKEND.blocks,
    max(PRIMARY_BAND),
    position=_property_inputs.final_prompt_position,
    delta=_answer_delta,
    multiplier=1.0,
):
    _direct_property = score_candidate_sequences(BACKEND, _property_inputs, PROPERTY_IDS)
with residual_intervention(
    BACKEND.blocks,
    max(PRIMARY_BAND),
    position=_identity_inputs.final_prompt_position,
    delta=_answer_delta,
    multiplier=1.0,
):
    _direct_identity = score_candidate_sequences(BACKEND, _identity_inputs, IDENTITY_IDS)

DIRECT_ANSWER_CONTROL = {
    "property": prediction_and_margin(_direct_property, "four")["prediction"],
    "identity": prediction_and_margin(_direct_identity, "cat")["prediction"],
}
print(f"inserting the `four` lens vector directly: legs -> {DIRECT_ANSWER_CONTROL['property']}, identity -> {DIRECT_ANSWER_CONTROL['identity']}")
assert DIRECT_ANSWER_CONTROL == {"property": "four", "identity": "bird"}
print()
print("The property answer moved WITHOUT the identity moving. A downstream change")
print("on its own is therefore not evidence of recomputation - the real study must")
print("report this control beside every downstream result.")


In [ ]:
# 10c. The completed pilot's baselines, kept available and kept LABELLED.
STEERING_BASELINES = {
    "control_kinds": list(causal.CONTROL_KINDS),
    "default_alphas": list(causal.DEFAULT_ALPHAS),
    "formula": "h' = h +- alpha * v_concept  (v_concept = V ReLU(mean_pos - mean_neg))",
    "family": "source_derived_jspace_steering",
}
COORDINATE_SWAP_CONTROLS = list(CONTROL_KINDS)
print("A - completed pilot (jlens.mmpilot.causal):")
print(f"    {STEERING_BASELINES['formula']}")
print(f"    controls {STEERING_BASELINES['control_kinds']}")
print()
print("B - planned coordinate swap (jlens.mmpilot.coordinate_swap):")
print("    h_patched = h + alpha * V (sigma(c) - c),  c = pinv(V) h")
print(f"    controls {COORDINATE_SWAP_CONTROLS}")
print()
print("`raw_residual_difference` and `source_derived_jspace_steering` appear in B's")
print("control list as BASELINES to compare against - never as coordinate swaps.")


## 11. The involution, measured

Exact exchange is its own inverse. Across a band it is recomputed from each
layer's own activation, so it only cancels to the extent that consecutive
layers agree — and in this synthetic world the carry blocks nearly commute with
the exchange, so they agree almost exactly. A real transformer's blocks do not,
but the real band still has to be chosen with this in mind rather than assumed
away.


In [ ]:
# 11. Identity outcome as a function of band length.
PARITY = band_parity_diagnostic(
    BACKEND, _identity_inputs, source=SOURCE, target=TARGET, max_band_length=4
)
for row in PARITY:
    print(f"band {str(row['band']):<14} length {row['band_length']} ({row['parity']:<4}) -> {row['prediction']}")
assert [row["swapped_to_target"] for row in PARITY] == [True, False, True, False]
print()
print("In THIS synthetic world an even-length band nearly cancels. That is a")
print("property of the method meeting blocks that commute with the exchange, not")
print("a bug, and it is a hazard the real layer band must be checked against.")


## 12. The spec, the fingerprint, atomic storage, and resume

The fingerprint names the intervention family, so a coordinate-swap run cannot
resume from a direction-steering run's directory — and the refusal says why
rather than only reporting a digest mismatch.


In [ ]:
# 12a. The versioned spec and the run fingerprint.
SPEC = build_spec(
    source=SOURCE,
    target=TARGET,
    layer_band=BAND,
    alpha=1.0,
    position_rule=PRIMARY_POSITION_RULE,
    control_kind="coordinate_swap",
    lens_checksums=mock_lens_checksums(BAND.layers),
    model_revision=MOCK_MODEL_REVISION,
    processor_revision=MOCK_PROCESSOR_REVISION,
    audio_protocol_fingerprint="mock-no-audio-protocol",
)
SWAP_FINGERPRINT = coordinate_swap_fingerprint(
    SPEC,
    alphas=(0.0, 0.5, 1.0, 2.0),
    controls=CONTROL_KINDS,
    control_config={"random_seed_base": 1000, "unrelated_pair": ["dog", "car"]},
)
print(json.dumps(SPEC.to_dict(), indent=2, sort_keys=True)[:1400])
print("...")
print()
print(f"spec digest {SPEC.digest}")
print(f"fingerprint keys: {sorted(SWAP_FINGERPRINT)}")


In [ ]:
# 12b. Atomic per-unit storage and fingerprint-gated resume, in a temp dir.
RUN_ROOT = Path(tempfile.mkdtemp(prefix="coordswap_mock_"))


def _store(root, fingerprint_overrides=None):
    intervention = dict(SWAP_FINGERPRINT)
    if fingerprint_overrides:
        intervention.update(fingerprint_overrides)
    return UnitStore(
        root,
        RunFingerprint(
            mode="mock_coordinate_swap",
            model_repo_id="mock",
            model_revision=MOCK_MODEL_REVISION,
            processor_revision=MOCK_PROCESSOR_REVISION,
            layers=tuple(BAND.layers),
            lens_checksum=MOCK_LENS_CHECKSUM,
            manifest_checksum="sha256:mock-manifest",
            split_id="mock-split",
            intervention_config=intervention,
        ),
    )


STORE = _store(RUN_ROOT / "run")
RESUME = {"first_open": STORE.open()}
for modality, result in IDENTITY_RESULTS.items():
    STORE.save(
        "intervention",
        f"identity__{modality}",
        {k: v for k, v in result.items() if k != "layer_stats"},
    )
RESUME["second_open"] = _store(RUN_ROOT / "run").open()
RESUME["units"] = len(_store(RUN_ROOT / "run").load_all("intervention"))
print(f"first open  {RESUME['first_open']}")
print(f"second open {RESUME['second_open']}  ({RESUME['units']} units reloaded)")
assert RESUME["first_open"] == "starting"
assert RESUME["second_open"] == "resuming"
assert RESUME["units"] == len(IDENTITY_RESULTS)


In [ ]:
# 12c. Every bound field refuses an incompatible resume.
RESUME_REFUSALS = {}
for label, override in (
    ("alpha set", {"alphas": [0.0, 1.0]}),
    ("position rule", {"position_rule": "final_prompt_token_only"}),
    ("layer band", {"layer_band": [1, 2]}),
    ("source token id", {"source_token_id": 999}),
    ("target token id", {"target_token_id": 999}),
    ("lens checksums", {"lens_checksums_by_layer": {"1": "x"}}),
    ("model revision", {"model_revision": "other"}),
    ("processor revision", {"processor_revision": "other"}),
    ("method version", {"coordinate_swap_method_version": "v99"}),
    ("condition threshold", {"condition_number_threshold": 1e9}),
    ("controls", {"controls": ["coordinate_swap"]}),
):
    try:
        _store(RUN_ROOT / "run", override).open()
    except IncompatibleStateError:
        RESUME_REFUSALS[label] = "refused"
    else:  # pragma: no cover - would be a real defect
        raise AssertionError(f"changing {label} did not refuse the resume")
for label in RESUME_REFUSALS:
    print(f"changing {label:<22} -> resume refused")


In [ ]:
# 12d. A coordinate-swap run cannot resume from a direction-steering run.
STEERING_DIR = RUN_ROOT / "steering_run"
UnitStore(
    STEERING_DIR,
    RunFingerprint(
        mode="pilot",
        model_repo_id="mock",
        model_revision=MOCK_MODEL_REVISION,
        processor_revision=MOCK_PROCESSOR_REVISION,
        layers=tuple(BAND.layers),
        lens_checksum=MOCK_LENS_CHECKSUM,
        manifest_checksum="sha256:mock-manifest",
        split_id="mock-split",
        intervention_config={
            "alphas": list(causal.DEFAULT_ALPHAS),
            "control_kinds": list(causal.CONTROL_KINDS),
        },
    ),
).open()

try:
    _store(STEERING_DIR).open()
except IncompatibleStateError as error:
    CROSS_FAMILY_REFUSAL = str(error).splitlines()[0]
else:  # pragma: no cover
    raise AssertionError("a steering run directory must not be resumable here")

_stored = json.loads((STEERING_DIR / "fingerprint.json").read_text(encoding="utf-8"))
try:
    assert_coordinate_swap_artifacts(_stored)
except CoordinateSwapError as error:
    CROSS_FAMILY_DIAGNOSIS = str(error)
else:  # pragma: no cover
    raise AssertionError("the family check must reject steering artifacts")

print(f"digest gate: {CROSS_FAMILY_REFUSAL}")
print()
print("family check:")
print(f"  {CROSS_FAMILY_DIAGNOSIS}")
assert_coordinate_swap_artifacts({"intervention_config": SWAP_FINGERPRINT})
print()
print("and the coordinate-swap fingerprint is accepted by the same check.")


## 13. Summary


In [ ]:
# 13. Collect everything this MOCK run established.
SUMMARY = {
    "mode": MODE,
    "method_version": METHOD_VERSION,
    "intervention_family": INTERVENTION_FAMILY,
    "ran_real_experiment": False,
    "loaded_gemma": False,
    "formula": "h_patched = h + alpha * V (sigma(c) - c),  c = pinv(V) h,  V = [v_source, v_target]",
    "lens_pair": {
        "cosine": BASIS_DIAGNOSTICS[PRIMARY_BAND[0]]["cosine"],
        "condition_number": BASIS_DIAGNOSTICS[PRIMARY_BAND[0]]["condition_number"],
        "numerical_rank": BASIS_DIAGNOSTICS[PRIMARY_BAND[0]]["numerical_rank"],
    },
    "algebra": {
        "alpha_zero_is_bit_exact": ALGEBRA["alpha_zero_is_bit_exact"],
        "double_swap_recovers": ALGEBRA["double_swap_recovers"],
        "coordinates_exchanged": ALGEBRA["coordinates_exchanged"],
        "orthogonal_drift": ALGEBRA["orthogonal_drift"],
    },
    "refusals": sorted(REFUSALS),
    "band": list(BAND.layers),
    "position_rule": PRIMARY_POSITION_RULE,
    "identity_predictions": {m: r["prediction"] for m, r in IDENTITY_RESULTS.items()},
    "property_predictions": {m: r["prediction"] for m, r in PROPERTY_RESULTS.items()},
    "layer_band_control": LAYER_BAND_CONTROL,
    "controls": CONTROLS,
    "direct_answer_control": DIRECT_ANSWER_CONTROL,
    "band_parity": [(row["band_length"], row["prediction"]) for row in PARITY],
    "resume": RESUME,
    "resume_refusals": sorted(RESUME_REFUSALS),
    "spec_digest": SPEC.digest,
}
STATUS = "MOCK_PASSED"
print(json.dumps(SUMMARY, indent=2, sort_keys=True))
shutil.rmtree(RUN_ROOT, ignore_errors=True)


---

# MOCK SUCCESS IS NOT SCIENTIFIC EVIDENCE

Everything above ran against a synthetic world that was **built to make the
algebra come out right**. The shared cross-modal concept vector, the
nonorthogonal lens pair, the reasoning layer that derives legs from identity —
all of it is stipulated, not measured.

What a passing MOCK run establishes:

* the implementation computes `h + alpha V (sigma(c) - c)` with `c = pinv(V) h`;
* `alpha = 0` is bit-exact, two swaps cancel, the coordinates are exchanged, and
  the orthogonal component does not move;
* the hooks patch every requested layer at every requested prompt position and
  never a teacher-forced candidate token;
* the refusals fire — rank-deficient pairs, ill-conditioned pairs, transposed
  bases, multi-token concepts, unvalidated layers, incompatible resumes, and a
  direction-steering run's artifacts;
* storage is atomic and resume is fingerprint-gated.

What it establishes about Gemma, about SpokenCOCO, and about any scientific
question: **nothing at all.**

In particular, **none** of the following is claimed anywhere in this notebook or
this repository under method B:

* that cross-modal identity replacement works;
* that a downstream property is recomputed rather than shortcut;
* that flexible generalization occurs;
* that multi-hop reasoning is recomputed consistently;
* that anything generalizes across evidence modalities.

Those require the real experiment, which cannot be designed until the running
earlier-layer calibration establishes whether an admissible contiguous band of
confirmed layers exists at all.
